## Salary Model Validation

In [1]:
from quant_slc_hedging.data_model import LoanModelInputs, SalaryModelInputs, SalaryGrowthType, salary_growth_amounts
from quant_slc_hedging.salary import SalaryModel
import numpy as np 
import pandas as pd

In [2]:
starting_salary=50_000
seed = 1234
years_remaining = 30
n_paths = 100_000

In [3]:
for growth_type, (rate, vol) in salary_growth_amounts.items():
    expected_salary = starting_salary * (1 + rate)**years_remaining
    print(f"For an expected rate of {rate * 100}% with vol of {vol * 100}% the expected salary is {expected_salary}")

For an expected rate of 1.0% with vol of 5.0% the expected salary is 67392.44576664531
For an expected rate of 5.0% with vol of 10.0% the expected salary is 216097.1187575334
For an expected rate of 10.0% with vol of 12.0% the expected salary is 872470.1134443224


## Check the average of the paths convergers to expected salary

In [4]:
results = []

for growth_type, (annual_growth, annual_vol) in salary_growth_amounts.items():

    config = SalaryModelInputs(
        starting_salary=starting_salary,
        salary_growth_dist=SalaryGrowthType(
            growth_type=growth_type
        ),
    )

    rng = np.random.default_rng(seed)

    model = SalaryModel(
        config=config,
        rng_gen=rng,
    )

    salary_paths = model.generate_salary_paths(
        n_paths=n_paths,
        n_months=12 * years_remaining,
    )

    for year in [1, 5, 10, 20, 30]:

        month = year * 12

        simulated_mean = salary_paths[:, month-1].mean()
        theoretical_mean = (
            starting_salary
            * (1 + annual_growth) ** year
        )

        results.append({
            "Growth Type": growth_type,
            "Year": year,
            "Simulated Mean": simulated_mean,
            "Theoretical Mean": theoretical_mean,
            "Difference": simulated_mean - theoretical_mean,
            "Difference %": (
                simulated_mean / theoretical_mean - 1
            ),
        })

results_df = pd.DataFrame(results)

In [5]:
results_df

,Growth Type,Year,Simulated Mean,Theoretical Mean,Difference,Difference %
0,Low,1,50453.625004,50500.000000,-46.374996,-0.000918
1,Low,5,52518.991052,52550.502505,-31.511453,-0.000600
2,Low,10,55161.703317,55231.106271,-69.402954,-0.001257
3,Low,20,60950.911069,61009.501997,-58.590928,-0.000960
4,Low,30,67416.790583,67392.445767,24.344817,0.000361
5,Medium,1,52277.138917,52500.000000,-222.861083,-0.004245
6,Medium,5,63585.082169,63814.078125,-228.995956,-0.003588
7,Medium,10,81053.150365,81444.731339,-391.580974,-0.004808
8,Medium,20,132092.833482,132664.885257,-572.051775,-0.004312
9,Medium,30,215776.150661,216097.118758,-320.968096,-0.001485


## Check convergence as n_paths increases

In [6]:
results = []
n_paths = [10_000, 25_000, 50_000, 100_000]

for n_path in n_paths:

    config = SalaryModelInputs(
        starting_salary=starting_salary,
        starting_loan_balance=60_000,
        salary_growth_dist=SalaryGrowthType(
            growth_type="Medium"
        ),
    )
    annual_growth, annual_vol = salary_growth_amounts[config.salary_growth_dist.growth_type]

    rng = np.random.default_rng(seed)

    model = SalaryModel(
        config=config,
        rng_gen=rng,
    )

    salary_paths = model.generate_salary_paths(
        n_paths=n_path,
        n_months=12 * years_remaining,
    )

    simulated_mean = salary_paths[:, -1].mean()
    theoretical_mean = starting_salary * (1+ annual_growth)**years_remaining
    results.append({
        "N_paths": n_path,
        "Simulated mean": simulated_mean,
        "Theoretical mean": theoretical_mean,
        "Error": abs(theoretical_mean - simulated_mean)

    })

results_df = pd.DataFrame(results)

TypeError: SalaryModelInputs.__init__() got an unexpected keyword argument 'starting_loan_balance'

In [7]:
results_df

,Growth Type,Year,Simulated Mean,Theoretical Mean,Difference,Difference %
0,Low,1,50453.625004,50500.000000,-46.374996,-0.000918
1,Low,5,52518.991052,52550.502505,-31.511453,-0.000600
2,Low,10,55161.703317,55231.106271,-69.402954,-0.001257
3,Low,20,60950.911069,61009.501997,-58.590928,-0.000960
4,Low,30,67416.790583,67392.445767,24.344817,0.000361
5,Medium,1,52277.138917,52500.000000,-222.861083,-0.004245
6,Medium,5,63585.082169,63814.078125,-228.995956,-0.003588
7,Medium,10,81053.150365,81444.731339,-391.580974,-0.004808
8,Medium,20,132092.833482,132664.885257,-572.051775,-0.004312
9,Medium,30,215776.150661,216097.118758,-320.968096,-0.001485
